In [1]:
import torch, torchaudio, random, itertools
import torch.nn as nn
import torch.optim as optim
import numpy as np, pandas as pd
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE, HOP_SECONDS = 44100, 0.010
HOP, N_FFT, N_MELS = int(HOP_SECONDS * SAMPLE_RATE), 2048, 128

class OnsetCRNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.mel   = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
        self.to_db = torchaudio.transforms.AmplitudeToDB()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,1)),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,1)),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
        )
        self.rnn = nn.GRU(128 * (N_MELS // 4), 128, num_layers=3, batch_first=True,
                          bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.state_head, self.onset_head, self.offset_head = nn.Linear(256, 2), nn.Linear(256, 1), nn.Linear(256, 1)
    def forward(self, x):
        s = self.cnn(self.to_db(self.mel(x)))
        B, C, M, T = s.shape
        s, _ = self.rnn(s.reshape(B, C*M, T).transpose(1, 2))
        s = self.drop(s)
        return self.state_head(s), self.onset_head(s).squeeze(-1), self.offset_head(s).squeeze(-1)

weight_files = sorted(Path("/kaggle/input").rglob("biggerCRNN_ext4_seed*.pt"))

TONAS = Path("/kaggle/input/datasets/alirahmatpour/vocalset-onset/tonas_upload")
def singer_of(stem):
    """'24-M1_AMairena-Martinete' -> 'AMairena'"""
    return stem.split("_", 1)[1].split("-")[0] if "_" in stem else stem

pairs = []
for wav in sorted(TONAS.rglob("*.wav")):
    notes = wav.parent / (wav.stem + ".notes.Corrected")
    if wav.name.startswith("._") or not notes.exists():
        continue
    pairs.append((wav, notes))

# ---- checks ----
print("device :", device, "| weights:", [w.name for w in weight_files], "(expect 3)")
print("clips with audio + notes:", len(pairs), "(expect 72)")
sing = pd.Series([singer_of(w.stem) for w, _ in pairs])
print("singers:", sing.nunique(), "| clips per singer (top 10):", sing.value_counts().head(10).to_dict())

device : cuda | weights: ['biggerCRNN_ext4_seed0.pt', 'biggerCRNN_ext4_seed1.pt', 'biggerCRNN_ext4_seed2.pt'] (expect 3)
clips with audio + notes: 72 (expect 72)
singers: 53 | clips per singer (top 10): {'AMairena': 5, 'JuanTalega': 4, 'Chocolate': 3, 'JAlmaden': 2, 'ChanoLobato': 2, 'MSimon': 2, 'MVargas': 2, 'Naranjito': 2, 'PdeLucia': 2, 'TPabon': 2}


In [2]:
songs = []
for wav, notes in pairs:
    wf, sr = torchaudio.load(str(wav))
    wave = wf.mean(0)
    if sr != SAMPLE_RATE:
        wave = torchaudio.functional.resample(wave, sr, SAMPLE_RATE)
    on, off = [], []
    for ln in notes.read_text(encoding="latin-1").strip().splitlines()[1:]:
        p = [x.strip() for x in ln.split(",")]
        if len(p) >= 2:
            on.append(float(p[0])); off.append(float(p[0]) + float(p[1]))
    songs.append({"key": wav.stem, "singer": singer_of(wav.stem), "style": wav.parent.name,
                  "sr": sr, "wave": wave, "on": np.array(on), "off": np.array(off)})
print("songs:", len(songs), "| sample rates:", sorted({s['sr'] for s in songs}),
      "| singers:", len({s['singer'] for s in songs}),
      "| styles:", pd.Series([s['style'] for s in songs]).value_counts().to_dict())
print("notes:", sum(len(s['on']) for s in songs))

def pick_peaks(prob, thr, min_gap=3):
    peaks = []
    for i in range(len(prob)):
        if prob[i] < thr: continue
        lo, hi = max(0, i-1), min(len(prob), i+2)
        if prob[i] == prob[lo:hi].max() and (not peaks or i - peaks[-1] >= min_gap):
            peaks.append(i)
    return set(peaks)

def f1_sets(pred_list, true_list, tol):
    tp = fp = fn = 0
    for pred, t_idx in zip(pred_list, true_list):
        matched = set()
        for ti in t_idx:
            hit = next((pi for pi in pred if abs(pi - ti) <= tol and pi not in matched), None)
            if hit is not None: tp += 1; matched.add(hit)
            else: fn += 1
        fp += len(set(pred) - matched)
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    return 2*p*r/(p+r) if p+r else 0.0

frames = [set((s["on"] / HOP_SECONDS).astype(int)) for s in songs]

print("\n=== HUMAN CEILING ===")
print("not available: TONAS has only one annotator")

rows, offsets = [], []
for w in weight_files:
    model = OnsetCRNN().to(device)
    model.load_state_dict(torch.load(w, map_location=device)); model.eval()
    with torch.no_grad():
        probs = [torch.sigmoid(model(s["wave"].view(1,1,-1).to(device))[1])[0].cpu().numpy() for s in songs]
    peaks08 = [pick_peaks(p, 0.8) for p in probs]
    best_thr = max([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
                   key=lambda th: f1_sets([pick_peaks(p, th) for p in probs], frames, 2))
    peaksB = [pick_peaks(p, best_thr) for p in probs]
    rows.append({"seed": w.stem[-1],
                 "F1_20": f1_sets(peaks08, frames, 2), "F1_50": f1_sets(peaks08, frames, 5),
                 "best_thr": best_thr, "bestF1_20": f1_sets(peaksB, frames, 2),
                 "bestF1_50": f1_sets(peaksB, frames, 5)})
    for pred, t in zip(peaks08, frames):
        t = np.array(sorted(t))
        if len(t) == 0: continue
        for pi in pred:
            dd = pi - t[np.argmin(np.abs(t - pi))]
            if abs(dd) <= 20: offsets.append(dd * 10)
    del model; torch.cuda.empty_cache()

zs = pd.DataFrame(rows)
print("\n=== ZERO-SHOT: VocalSet model on TONAS (mean ± std over 3 seeds) ===")
for c in ["F1_20", "F1_50", "bestF1_20", "bestF1_50"]:
    print(f"{c:10s}: {zs[c].mean():.3f} ± {zs[c].std(ddof=1):.3f}")
print("best thresholds picked:", sorted(zs.best_thr.unique()))
off_arr = np.array(offsets)
print(f"\ntiming offset (model - human): median {np.median(off_arr):+.0f} ms | within ±20ms: {np.mean(np.abs(off_arr)<=20):.0%}")

songs: 72 | sample rates: [44100] | singers: 53 | styles: {'Martinetes1': 36, 'Martinetes2': 20, 'Deblas': 16}
notes: 2983

=== HUMAN CEILING ===
not available: TONAS has only one annotator

=== ZERO-SHOT: VocalSet model on TONAS (mean ± std over 3 seeds) ===
F1_20     : 0.145 ± 0.038
F1_50     : 0.409 ± 0.043
bestF1_20 : 0.186 ± 0.015
bestF1_50 : 0.435 ± 0.013
best thresholds picked: [np.float64(0.1)]

timing offset (model - human): median +40 ms | within ±20ms: 20%


In [3]:
# ---- split by singer: about 1/3 of clips to test, no singer on both sides ----
rng = random.Random(0)
singers = sorted({s["singer"] for s in songs}); rng.shuffle(singers)
test_singers, n_test = set(), 0
for sg in singers:
    if n_test >= len(songs) // 3: break
    test_singers.add(sg); n_test += sum(s["singer"] == sg for s in songs)
test_i = [i for i, s in enumerate(songs) if s["singer"] in test_singers]
pool_i = [i for i, s in enumerate(songs) if s["singer"] not in test_singers]
rng.shuffle(pool_i)                                            # first N = the N training clips
print("test singers:", len(test_singers), "| test clips:", len(test_i), "| training pool:", len(pool_i))
print("test styles:", pd.Series([songs[i]["style"] for i in test_i]).value_counts().to_dict())

SOFT_WIDTH = 2
def make_targets(s):
    nfr = 1 + s["wave"].shape[0] // HOP
    st = np.zeros(nfr, np.int64); on_t = np.zeros(nfr, np.float32); off_t = np.zeros(nfr, np.float32)
    def bump(arr, c):
        for d in range(-SOFT_WIDTH, SOFT_WIDTH + 1):
            j = c + d
            if 0 <= j < nfr: arr[j] = max(arr[j], np.exp(-(d*d) / (2 * (SOFT_WIDTH/2)**2)))
    for a, b in zip(s["on"], s["off"]):
        fa, fb = int(a / HOP_SECONDS), int(b / HOP_SECONDS)
        st[fa:fb] = 1; bump(on_t, fa); bump(off_t, fb)
    return torch.tensor(st), torch.tensor(on_t), torch.tensor(off_t)
tg = [make_targets(s) for s in songs]

def train_model(start, train_idx, seed, epochs, lr, crop=1500):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = OnsetCRNN().to(device)
    if start is not None:
        model.load_state_dict(torch.load(start, map_location=device))
    if not train_idx: return model
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    ce, bce = nn.CrossEntropyLoss(), nn.BCEWithLogitsLoss(pos_weight=torch.tensor(20.0, device=device))
    for ep in range(epochs):
        model.train()
        for i in random.sample(train_idx, len(train_idx)):
            st, on, off = tg[i]
            nfr = len(st); a = np.random.randint(0, max(1, nfr - crop)); b = min(nfr, a + crop)
            ps, po, pf = model(songs[i]["wave"][a*HOP : b*HOP].view(1, 1, -1).to(device))
            F = min(ps.shape[1], b - a)
            loss = (ce(ps[0, :F], st[a:a+F].to(device)) + bce(po[0, :F], on[a:a+F].to(device))
                    + bce(pf[0, :F], off[a:a+F].to(device)))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model):
    model.eval()
    with torch.no_grad():
        probs = [torch.sigmoid(model(songs[i]["wave"].view(1,1,-1).to(device))[1])[0].cpu().numpy() for i in test_i]
    t = [frames[i] for i in test_i]
    out = {}
    for thr_name, thr in [("", 0.8), ("best_", None)]:
        if thr is None:
            thr = max([0.3,0.4,0.5,0.6,0.7,0.8,0.9],
                      key=lambda th: f1_sets([pick_peaks(p, th) for p in probs], t, 5))
        peaks = [pick_peaks(p, thr) for p in probs]
        for tol_name, tol in [("20", 2), ("50", 5)]:
            out[f"{thr_name}F1_{tol_name}"] = f1_sets(peaks, t, tol)
    return out

# ---- SAME training for both: 40 rounds, same step size (fair comparison) ----
SETUPS = {"pretrained": dict(epochs=40, lr=1e-3),     # starts from VocalSet weights
          "scratch":    dict(epochs=40, lr=1e-3)}     # starts empty
N_LIST = sorted({0, 3, 9, 18, len(pool_i)})
rows = []
for N in N_LIST:
    for k, w in enumerate(weight_files):
        for name, cfg in SETUPS.items():
            if N == 0 and name == "scratch": continue
            model = train_model(w if name == "pretrained" else None, pool_i[:N], seed=k, **cfg)
            r = evaluate(model); r.update({"N": N, "start": name, "seed": k}); rows.append(r)
            print(f"N={N:2d} | {name:10s} | seed {k} | ±20 {r['F1_20']:.3f} | ±50 {r['F1_50']:.3f}"
                  f" | best thr ±50 {r['best_F1_50']:.3f}")
            del model; torch.cuda.empty_cache()

res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/onset_V3_tonas_zeroshot_vs_finetune.csv", index=False)
print("\n=== TONAS test singers — mean ± std over 3 seeds (both: 40 rounds, lr 1e-3) ===")
for (N, start), g in res.groupby(["N", "start"]):
    print(f"N={N:2d} {start:10s} | ±20 {g.F1_20.mean():.3f} ± {g.F1_20.std(ddof=1):.3f}"
          f" | ±50 {g.F1_50.mean():.3f} ± {g.F1_50.std(ddof=1):.3f} | best-thr ±50 {g.best_F1_50.mean():.3f}")

test singers: 15 | test clips: 24 | training pool: 48
test styles: {'Martinetes1': 12, 'Deblas': 7, 'Martinetes2': 5}
N= 0 | pretrained | seed 0 | ±20 0.139 | ±50 0.411 | best thr ±50 0.453
N= 0 | pretrained | seed 1 | ±20 0.130 | ±50 0.394 | best thr ±50 0.471
N= 0 | pretrained | seed 2 | ±20 0.201 | ±50 0.486 | best thr ±50 0.488
N= 3 | pretrained | seed 0 | ±20 0.466 | ±50 0.722 | best thr ±50 0.725
N= 3 | scratch    | seed 0 | ±20 0.451 | ±50 0.723 | best thr ±50 0.725
N= 3 | pretrained | seed 1 | ±20 0.518 | ±50 0.741 | best thr ±50 0.746
N= 3 | scratch    | seed 1 | ±20 0.457 | ±50 0.697 | best thr ±50 0.707
N= 3 | pretrained | seed 2 | ±20 0.491 | ±50 0.755 | best thr ±50 0.755
N= 3 | scratch    | seed 2 | ±20 0.459 | ±50 0.679 | best thr ±50 0.693
N= 9 | pretrained | seed 0 | ±20 0.517 | ±50 0.744 | best thr ±50 0.746
N= 9 | scratch    | seed 0 | ±20 0.499 | ±50 0.729 | best thr ±50 0.729
N= 9 | pretrained | seed 1 | ±20 0.528 | ±50 0.746 | best thr ±50 0.749
N= 9 | scratch    